# NLP Lab 4: GPT-2 (124M) Pipeline From Scratch

A clean, minimal implementation of the **GPT-2 (124M)** architecture in PyTorch.
We build every component from scratch, load the official pretrained weights, and generate text.


## 1. Imports and Hyperparameters


In [1]:
import math
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer
from huggingface_hub import hf_hub_download
import safetensors.torch as st

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# GPT-2 (124M variant) architecture settings
GPT_CONFIG = {
    'vocab_size':     50257,   # BPE vocabulary size
    'context_length': 1024,    # Maximum sequence length
    'emb_dim':        768,     # Embedding dimension
    'n_heads':        12,      # Number of attention heads (768 / 12 = 64 dims/head)
    'n_layers':       12,      # Number of transformer blocks
    'drop_rate':      0.1,     # Dropout probability
    'qkv_bias':       True,    # GPT-2 uses bias in QKV projections
}


Device: cpu


## 2. Tokenization (BPE)

GPT-2 uses Byte-Pair Encoding (BPE) with a 50,257-token vocabulary.


In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

text = 'Alan Turing was a brilliant mathematician.'
tokens = tokenizer.encode(text)
print('Text   :', text)
print('Tokens :', tokens)
print('Decoded:', [tokenizer.decode([t]) for t in tokens])


Text   : Alan Turing was a brilliant mathematician.
Tokens : [36235, 39141, 373, 257, 10457, 48251, 13]
Decoded: ['Alan', ' Turing', ' was', ' a', ' brilliant', ' mathematician', '.']


## 3. Architecture Components

### 3.1 Multi-Head Causal Self-Attention
Computes attention across 12 heads. A lower-triangular causal mask ensures tokens only attend to past and current positions, never the future.


In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, context_length, drop_rate=0.1, qkv_bias=True):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads

        # Combined projection for Query, Key, and Value
        self.qkv_proj = nn.Linear(emb_dim, 3 * emb_dim, bias=qkv_bias)
        self.out_proj = nn.Linear(emb_dim, emb_dim)
        self.attn_drop = nn.Dropout(drop_rate)
        self.resid_drop = nn.Dropout(drop_rate)

        # Causal mask: lower-triangular matrix of 1s
        mask = torch.tril(torch.ones(context_length, context_length))
        self.register_buffer('mask', mask.view(1, 1, context_length, context_length))

    def forward(self, x):
        B, T, C = x.shape

        # Project and split into Q, K, V -> (B, n_heads, T, head_dim)
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention + causal masking
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        # Aggregate values and project back to emb_dim
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(out))


### 3.2 Feed-Forward Network (MLP)
A two-layer position-wise MLP ($768 \to 3072 \to 768$) with GeLU activation.


In [4]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, drop_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, 4 * emb_dim)
        self.gelu = nn.GELU(approximate='tanh')
        self.fc2 = nn.Linear(4 * emb_dim, emb_dim)
        self.drop = nn.Dropout(drop_rate)

    def forward(self, x):
        return self.drop(self.fc2(self.gelu(self.fc1(x))))


### 3.3 Transformer Block
Combines Attention and MLP using **Pre-Layer Normalization** and residual skip-connections.


In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg['emb_dim'])
        self.attn = CausalSelfAttention(
            emb_dim=cfg['emb_dim'],
            n_heads=cfg['n_heads'],
            context_length=cfg['context_length'],
            drop_rate=cfg['drop_rate'],
            qkv_bias=cfg['qkv_bias']
        )
        self.ln2 = nn.LayerNorm(cfg['emb_dim'])
        self.mlp = FeedForward(cfg['emb_dim'], drop_rate=cfg['drop_rate'])

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


## 4. Full GPT-2 Architecture

Combines token embeddings, learned positional embeddings, 12 transformer blocks, final LayerNorm, and a linear head with **weight tying**.


In [6]:
class GPT2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop = nn.Dropout(cfg['drop_rate'])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.ln_f = nn.LayerNorm(cfg['emb_dim'])
        self.head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

        # Weight tying: share embedding weights with output head
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(0, T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(positions))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

model = GPT2Model(GPT_CONFIG).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,} ({total_params/1e6:.1f}M)')


Total parameters: 124,439,808 (124.4M)


## 5. Text Generation Pipeline

Generates text autoregressively. Supports both **greedy decoding** (deterministic) and **temperature + Top-K sampling**.


In [7]:
def generate(model, prompt, max_new_tokens=35, temperature=0.8, top_k=40):
    """Autoregressively generate text from a prompt."""
    model.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            cond_ids = input_ids[:, -GPT_CONFIG['context_length']:]
            logits = model(cond_ids)[:, -1, :]

            if temperature == 0.0:
                next_token = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    top_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits = torch.where(logits < top_vals[:, -1].unsqueeze(-1), float('-inf'), logits)
                probs = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

            input_ids = torch.cat([input_ids, next_token], dim=-1)

    return tokenizer.decode(input_ids[0].tolist())


## 6. Loading Pretrained Weights

We load OpenAI's official 124M weights via `safetensors` from Hugging Face Hub and map them into our model.


In [8]:
def load_pretrained_weights(model):
    print('Downloading official GPT-2 (124M) weights from Hugging Face...')
    weights_path = hf_hub_download('gpt2', 'model.safetensors')
    hf_sd = st.load_file(weights_path)
    our_sd = model.state_dict()

    mapping = {
        'tok_emb.weight': 'wte.weight',
        'pos_emb.weight': 'wpe.weight',
        'ln_f.weight':    'ln_f.weight',
        'ln_f.bias':      'ln_f.bias',
    }
    for i in range(GPT_CONFIG['n_layers']):
        mapping[f'blocks.{i}.ln1.weight']           = f'h.{i}.ln_1.weight'
        mapping[f'blocks.{i}.ln1.bias']             = f'h.{i}.ln_1.bias'
        mapping[f'blocks.{i}.attn.qkv_proj.weight'] = f'h.{i}.attn.c_attn.weight'
        mapping[f'blocks.{i}.attn.qkv_proj.bias']   = f'h.{i}.attn.c_attn.bias'
        mapping[f'blocks.{i}.attn.out_proj.weight'] = f'h.{i}.attn.c_proj.weight'
        mapping[f'blocks.{i}.attn.out_proj.bias']   = f'h.{i}.attn.c_proj.bias'
        mapping[f'blocks.{i}.ln2.weight']           = f'h.{i}.ln_2.weight'
        mapping[f'blocks.{i}.ln2.bias']             = f'h.{i}.ln_2.bias'
        mapping[f'blocks.{i}.mlp.fc1.weight']       = f'h.{i}.mlp.c_fc.weight'
        mapping[f'blocks.{i}.mlp.fc1.bias']         = f'h.{i}.mlp.c_fc.bias'
        mapping[f'blocks.{i}.mlp.fc2.weight']       = f'h.{i}.mlp.c_proj.weight'
        mapping[f'blocks.{i}.mlp.fc2.bias']         = f'h.{i}.mlp.c_proj.bias'

    linear_keys = {'qkv_proj', 'out_proj', 'fc1', 'fc2'}
    with torch.no_grad():
        for our_k, hf_k in mapping.items():
            tensor = hf_sd[hf_k]
            # Transpose Conv1D weights to match PyTorch Linear layout
            if any(k in our_k for k in linear_keys) and 'weight' in our_k:
                tensor = tensor.t()
            our_sd[our_k].copy_(tensor)

    model.load_state_dict(our_sd)
    print('Successfully loaded weights!')
    return model

model = load_pretrained_weights(model)


Successfully loaded weights!


## 7. Generating Text

Let's test our trained model on a few prompts using different decoding settings.


In [9]:
# 1. Deterministic completion (Greedy)
print('--- Greedy Decoding (temperature=0.0) ---')
print(generate(model, 'Alan Turing was a', max_new_tokens=25, temperature=0.0))
print()

# 2. Natural completion (Temperature + Top-K)
print('--- Sampling (temperature=0.8, top_k=40) ---')
print(generate(model, 'In a distant galaxy, astronomers discovered', max_new_tokens=35, temperature=0.8, top_k=40))
print()

# 3. Story prompt
print('--- Story Prompt ---')
print(generate(model, 'Once upon a time in a small village nestled between mountains,', max_new_tokens=35, temperature=0.8, top_k=40))


--- Greedy Decoding (temperature=0.0) ---


Alan Turing was a brilliant mathematician, and he was a great friend of mine. He was a great friend of mine. He was a great friend

--- Sampling (temperature=0.8, top_k=40) ---


In a distant galaxy, astronomers discovered the first known supernova remnant known to contain an exoplanet.

We know about a few planets, but what's unique is the fact that they're at the

--- Story Prompt ---


Once upon a time in a small village nestled between mountains, the children of the village became familiar with the strange and dark world of the village.

The local inhabitants of the village had been waiting for a few days on the day
